# Proyecto Final- Agente SQL Conversacional
Nombre del Estudiante: Sergio Nuñez Lazo

Fecha de Entrega: 08/03/2026

Curso: Fundamentos de LLM y Datos


##  1. Introducción y Contexto del Proyecto

### 1.1 Descripción del Dominio
He elegido trabajar con datos relacionados a Formula 1 (F1). Los datos cuentan con los resultados de cada carrera y e limitado los datos a los años 2022, 2023, 2024 y 2025. Como fan de la F1, estos datos contienen información de una gran cantidad de carreras que e visto.

### 1.2 Fuente de los Datos
Obtuve los datos usando el API de FastF1. Para obtener los datos se utilizo el siguiente codigo.

In [24]:
# !pip install fastf1

In [25]:
##Imports
#import fastf1
#import pandas as pd
#
#
#years = [2022,2023,2024,2025] #Años seleccionados
#races = pd.DataFrame() #Dataset temporal donde se guardan los datos
#
#for year in years: 
#    schedule = fastf1.get_event_schedule(year) #Obtener los datos de cada evento
#    for name in schedule.EventName:
#        if name.startswith("Pre-"): #Ignorar las carreras de Pretemporada
#            continue
#        session = fastf1.get_session(year, name, 'race') #Obtener datos de una carrera en especifico
#        session.load() #Cargar datos
# 
#        #Añadir a resultados el año y nombre de la carrera 
#        session.results['year'] = year
#        session.results['raceName'] = name
#        temp = pd.DataFrame(session.results) #Dataset temporal
#        races = pd.concat([races,temp], ignore_index= True) #Añadir resultados a races
#
# #Pasar datos a csv
# races.to_csv('races.csv',index=False) 
# races.head(10)

### 1.3 Descripción General del Dataset


El dataset contiene 1918 registros correspondiente a los resultados de cada conductor en cada carrera entre los años 2022 y 2025.

 Cada fila representa el resultado de un conductor X (e.g. Max Verstappen) en una carrera Y (e.g. GP de Miami 2025). Las filas contienen datos del conductor (DriverNumber, BroadcastName, Abbreviation, DriverId, TeamName TeamColor, TeamId, FirstName, LastName, FullName, HeadshotUrl y CountryCode), datos del desempeño en la carrera (Position,ClassifiedPosition, GridPosition, Q1, Q2, Q3, Time, Status, Points y Laps) y datos relacionados a la carrera (year y raceName).

### 1.4 Preguntas de Negocio Relevantes
- ¿Que conductor gano mas puntos durante los años 2022, 2023, 2024 y 2025?
- ¿Que conductor obtuvo la mayor racha de victorias?
- ¿Que conductores ganaron el mismo GP 2 años seguidos?
- ¿Que conductor corrio la mayor cantidad de vueltas?
- ¿Que tipo de fallo o accidente causa mayor cantidad de DNFs?


## 2. Exploración Inicial de los Datos

### 2.1 Instalación de Dependencias

In [1]:
# Instala todas las librerías necesarias
#!pip install -q pandas numpy sqlalchemy langchain langchain-openai langchain-community

### 2.2 Importación de Librerías

In [2]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine
import os
from datetime import datetime
# Configuración para visualización más legible de DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)


### 2.3 Carga del Dataset Original

In [3]:
# Carga tus datos desde CSV, Excel, o el formato que tengas
# Ajusta la ruta y el método de carga según tu caso
df_original = pd.read_csv('races.csv')
# Muestra las primeras filas para verificar la carga
df_original.head()


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,year,raceName
0,16,C LECLERC,LEC,leclerc,Ferrari,ed1c24,ferrari,Charles,Leclerc,Charles Leclerc,https://www.formula1.com/content/dam/fom-websi...,MON,1.0,1,1.0,NaN,NaN,NaN,0 days 01:37:33.584000,Finished,26.0,57.0,2022,Bahrain Grand Prix
1,55,C SAINZ,SAI,sainz,Ferrari,ed1c24,ferrari,Carlos,Sainz,Carlos Sainz,https://www.formula1.com/content/dam/fom-websi...,ESP,2.0,2,3.0,NaN,NaN,NaN,0 days 00:00:05.598000,Finished,18.0,57.0,2022,Bahrain Grand Prix
2,44,L HAMILTON,HAM,hamilton,Mercedes,6cd3bf,mercedes,Lewis,Hamilton,Lewis Hamilton,https://www.formula1.com/content/dam/fom-websi...,GBR,3.0,3,5.0,NaN,NaN,NaN,0 days 00:00:09.675000,Finished,15.0,57.0,2022,Bahrain Grand Prix
3,63,G RUSSELL,RUS,russell,Mercedes,6cd3bf,mercedes,George,Russell,George Russell,https://www.formula1.com/content/dam/fom-websi...,GBR,4.0,4,9.0,NaN,NaN,NaN,0 days 00:00:11.211000,Finished,12.0,57.0,2022,Bahrain Grand Prix
4,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,b6babd,haas,Kevin,Magnussen,Kevin Magnussen,NaN,DEN,5.0,5,7.0,NaN,NaN,NaN,0 days 00:00:14.754000,Finished,10.0,57.0,2022,Bahrain Grand Prix


### 2.4 Exploración Estadística Básica

In [4]:
# Información general sobre el DataFrame
print("Información del Dataset:")
print(df_original.info())
print("\n" + "="*80 + "\n")
# Resumen estadístico de columnas numéricas
print("Resumen Estadístico:")
print(df_original.describe())
print("\n" + "="*80 + "\n")
# Verifica valores nulos
print("Valores Nulos por Columna:")
print(df_original.isnull().sum())
print("\n" + "="*80 + "\n")
# Para columnas categóricas, muestra los valores únicos
print("Valores Únicos en Columnas Categóricas:")
for col in df_original.select_dtypes(include=['object']).columns:
    print(f"\n{col}: {df_original[col].nunique()} valores únicos")
    print(f"Ejemplos: {df_original[col].value_counts().head(5).to_dict()}")

Información del Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1918 entries, 0 to 1917
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   DriverNumber        1918 non-null   int64  
 1   BroadcastName       1916 non-null   object 
 2   Abbreviation        1918 non-null   object 
 3   DriverId            1918 non-null   object 
 4   TeamName            1918 non-null   object 
 5   TeamColor           1916 non-null   object 
 6   TeamId              1918 non-null   object 
 7   FirstName           1918 non-null   object 
 8   LastName            1918 non-null   object 
 9   FullName            1918 non-null   object 
 10  HeadshotUrl         1842 non-null   object 
 11  CountryCode         1437 non-null   object 
 12  Position            1916 non-null   float64
 13  ClassifiedPosition  1918 non-null   object 
 14  GridPosition        1916 non-null   float64
 15  Q1                  0 non-null

### 2.5 Análisis Inicial y Observaciones

Tras el análisis inicial se encuentra que las siguientes observaciones:

- Las columnas "Q1", "Q2" y "Q3" solo contienen valores nulos, esto es causado por como se obtienen los datos usando fast f1. Debido a que solo se uso los datos de las carreras, el API no envio los tiempos de las clasificaciones o "qualis". 
- La columna "Time" contiene valores nulos, pero esto se debe a que no todos los conductores terminan todas las carreras, ya que aveces son descalificados debido a accidentes o problemas con sus carros.

## 3. Preprocesamiento y Limpieza de Datos

### 3.1 Manejo de Valores Nulos

In [5]:
# Crea una copia del DataFrame para preprocesamiento
df_procesado = df_original.copy()

qualis = ['Q1','Q2','Q3']
df_procesado = df_procesado.drop(qualis, axis=1)
df_procesado = df_procesado.drop('HeadshotUrl', axis=1)
df_procesado = df_procesado.drop('CountryCode', axis=1)
df_procesado['Time'].fillna('DNF', inplace=True)

print("Valores nulos después de limpieza:")
print(df_procesado.isnull().sum())

Valores nulos después de limpieza:
DriverNumber          0
BroadcastName         2
Abbreviation          0
DriverId              0
TeamName              0
TeamColor             2
TeamId                0
FirstName             0
LastName              0
FullName              0
Position              2
ClassifiedPosition    0
GridPosition          2
Time                  0
Status                0
Points                0
Laps                  0
year                  0
raceName              0
dtype: int64


C:\Users\Sergio\AppData\Local\Temp\ipykernel_25668\1586251271.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_procesado['Time'].fillna('DNF', inplace=True)


In [6]:
df_original[df_original['BroadcastName'].isna()]

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,year,raceName
39,47,NaN,MSC,mick_schumacher,Haas F1 Team,NaN,haas,Mick,Schumacher,Mick Schumacher,NaN,NaN,NaN,W,NaN,NaN,NaN,NaN,NaN,Withdrew,0.0,0.0,2022,Saudi Arabian Grand Prix
739,18,NaN,STR,stroll,Aston Martin,NaN,aston_martin,Lance,Stroll,Lance Stroll,NaN,NaN,NaN,W,NaN,NaN,NaN,NaN,NaN,Withdrew,0.0,0.0,2023,Singapore Grand Prix


In [7]:
df_procesado = df_procesado.dropna(subset=["BroadcastName"])

print("Valores nulos después de limpieza:")
print(df_procesado.isnull().sum())

Valores nulos después de limpieza:
DriverNumber          0
BroadcastName         0
Abbreviation          0
DriverId              0
TeamName              0
TeamColor             0
TeamId                0
FirstName             0
LastName              0
FullName              0
Position              0
ClassifiedPosition    0
GridPosition          0
Time                  0
Status                0
Points                0
Laps                  0
year                  0
raceName              0
dtype: int64


- Se eliminaron las columnas "Q1","Q2" y "Q3" ya que solo conteninan valores nulos
- Se elimino la columna "HeadshotUrl" por que contenia valores nulos, y los valores que no eran nulos contenian un link hacia la foto del conductor, lo cual no es util para este trabajo ya que no aporta a los resultados de las carreras
- Al igual que el caso anterior, se elimino la columna "CountryCode" por que contenia valores nulos, y la nacionalidad del conductor no aporta en el proyecto
- Los valores nulos de la columna "Time" se cambiaron a "DNF" que significa "Did Not Finish" que se utiliza cuando un piloto no termina la carrera
- Se eliminaron las filas que contenian valores nulos en "BroadcastName" (estas filas tambien tenian valores nulos en "TeamColor", "Position" y "GridPosition") ya que son filas que muestran información sobre conductores que no participaron en el Grand Prix que aparece en su fila. Estos casos son los siguientes:
    - En el Grand Prix de Arabia Saudi 2022, Mick Shumacher se retiro debido a un choque en las carrera clasificatoria Q2.
    - En el Grand Prix de Singapur 2023, Lance Stroll se retiro debido a un choque en la carrera clasificatoria Q1.



### 3.2 Corrección de Tipos de Datos

In [8]:
df_procesado['Position'] = df_procesado['Position'].astype('int64')
df_procesado['GridPosition'] = df_procesado['GridPosition'].astype('int64')
df_procesado['Points'] = df_procesado['Points'].astype('int64')
df_procesado['Laps'] = df_procesado['Laps'].astype('int64')

# Verifica los tipos finales
print("Tipos de datos después de conversión:")
print(df_procesado.dtypes)


Tipos de datos después de conversión:
DriverNumber           int64
BroadcastName         object
Abbreviation          object
DriverId              object
TeamName              object
TeamColor             object
TeamId                object
FirstName             object
LastName              object
FullName              object
Position               int64
ClassifiedPosition    object
GridPosition           int64
Time                  object
Status                object
Points                 int64
Laps                   int64
year                   int64
raceName              object
dtype: object


Se convirtio las columnas "Position", "GridPosition", "Points" y "Laps" a int por que no era necesario que sean float, ya que todos sus valores no tenian decimales. Esto se debe a que no puedes quedar en posiciones decimales (e.g. Quedar en el puesto 1.5), empezar en puestos decimales (e.g. Tu posición en el grid no puede ser 1.5), obtener puntos con decimales (solo se pueden conseguir 25, 18, 15, 12, 10, 8, 6, 4, 2 o 1. Ademas de obtener 1 punto adicional si obtienes la vuelta mas rapida de la carrera, con excepcion de las carreras de 2025) y el dataset solo cuenta el numero de vueltas completadas, por lo que si un conductor abandona la carrera a la mitad de una vuelta no se le considera como x.5 vueltas.

### 3.3 Normalización y Estandarización

In [9]:
print("Valores únicos:")
for col in df_procesado.select_dtypes(include=['object', 'category']).columns:
    print(f"\n{col}: {df_procesado[col].nunique()} valores únicos")
    print(df_procesado[col].value_counts().head(10))

Valores únicos:

BroadcastName: 32 valores únicos
BroadcastName
C LECLERC       96
L HAMILTON      96
G RUSSELL       96
Y TSUNODA       96
L NORRIS        96
F ALONSO        96
M VERSTAPPEN    96
P GASLY         96
A ALBON         95
E OCON          95
Name: count, dtype: int64

Abbreviation: 31 valores únicos
Abbreviation
LEC    96
HAM    96
RUS    96
ALO    96
NOR    96
TSU    96
VER    96
GAS    96
ALB    95
OCO    95
Name: count, dtype: int64

DriverId: 31 valores únicos
DriverId
leclerc           96
hamilton          96
russell           96
alonso            96
norris            96
tsunoda           96
max_verstappen    96
gasly             96
albon             95
ocon              95
Name: count, dtype: int64

TeamName: 13 valores únicos
TeamName
Ferrari            192
Mercedes           192
Alpine             192
Red Bull Racing    192
McLaren            192
Williams           191
Haas F1 Team       191
Aston Martin       190
Kick Sauber        104
Alfa Romeo          88
Name: 

In [10]:
df_procesado['Time'] = df_procesado['Time'].apply(lambda x: x.split(" ")[-1] if x != 'DNF' else x)
df_procesado.head(20)

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Laps,year,raceName
0,16,C LECLERC,LEC,leclerc,Ferrari,ed1c24,ferrari,Charles,Leclerc,Charles Leclerc,1,1,1,01:37:33.584000,Finished,26,57,2022,Bahrain Grand Prix
1,55,C SAINZ,SAI,sainz,Ferrari,ed1c24,ferrari,Carlos,Sainz,Carlos Sainz,2,2,3,00:00:05.598000,Finished,18,57,2022,Bahrain Grand Prix
2,44,L HAMILTON,HAM,hamilton,Mercedes,6cd3bf,mercedes,Lewis,Hamilton,Lewis Hamilton,3,3,5,00:00:09.675000,Finished,15,57,2022,Bahrain Grand Prix
3,63,G RUSSELL,RUS,russell,Mercedes,6cd3bf,mercedes,George,Russell,George Russell,4,4,9,00:00:11.211000,Finished,12,57,2022,Bahrain Grand Prix
4,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,b6babd,haas,Kevin,Magnussen,Kevin Magnussen,5,5,7,00:00:14.754000,Finished,10,57,2022,Bahrain Grand Prix
5,77,V BOTTAS,BOT,bottas,Alfa Romeo,b12039,alfa,Valtteri,Bottas,Valtteri Bottas,6,6,6,00:00:16.119000,Finished,8,57,2022,Bahrain Grand Prix
6,31,E OCON,OCO,ocon,Alpine,2293d1,alpine,Esteban,Ocon,Esteban Ocon,7,7,11,00:00:19.423000,Finished,6,57,2022,Bahrain Grand Prix
7,22,Y TSUNODA,TSU,tsunoda,AlphaTauri,4e7c9b,alphatauri,Yuki,Tsunoda,Yuki Tsunoda,8,8,16,00:00:20.386000,Finished,4,57,2022,Bahrain Grand Prix
8,14,F ALONSO,ALO,alonso,Alpine,2293d1,alpine,Fernando,Alonso,Fernando Alonso,9,9,8,00:00:22.390000,Finished,2,57,2022,Bahrain Grand Prix
9,24,G ZHOU,ZHO,zhou,Alfa Romeo,b12039,alfa,Guanyu,Zhou,Guanyu Zhou,10,10,15,00:00:23.064000,Finished,1,57,2022,Bahrain Grand Prix


In [11]:
df_procesado['Race'] = df_procesado['year'].astype(str) + ' ' + df_procesado['raceName'].astype(str)
df_procesado = df_procesado.drop('year', axis=1)
df_procesado = df_procesado.drop('raceName', axis=1)
df_procesado.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Laps,Race
0,16,C LECLERC,LEC,leclerc,Ferrari,ed1c24,ferrari,Charles,Leclerc,Charles Leclerc,1,1,1,01:37:33.584000,Finished,26,57,2022 Bahrain Grand Prix
1,55,C SAINZ,SAI,sainz,Ferrari,ed1c24,ferrari,Carlos,Sainz,Carlos Sainz,2,2,3,00:00:05.598000,Finished,18,57,2022 Bahrain Grand Prix
2,44,L HAMILTON,HAM,hamilton,Mercedes,6cd3bf,mercedes,Lewis,Hamilton,Lewis Hamilton,3,3,5,00:00:09.675000,Finished,15,57,2022 Bahrain Grand Prix
3,63,G RUSSELL,RUS,russell,Mercedes,6cd3bf,mercedes,George,Russell,George Russell,4,4,9,00:00:11.211000,Finished,12,57,2022 Bahrain Grand Prix
4,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,b6babd,haas,Kevin,Magnussen,Kevin Magnussen,5,5,7,00:00:14.754000,Finished,10,57,2022 Bahrain Grand Prix


Los valores ya cuentan con un formato estandarizado por los autores de FastF1, pero se arreglo los valores temporales en "Time", ya que se elimino "days" ya que ninguna carrera dura mas de 1 dia. Tambien se junto "year" y "raceName".

### 3.4 Eliminación de Duplicados

In [12]:
duplicados = df_procesado[df_procesado.duplicated(keep=False)]
duplicados.head(10)

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Laps,Race
1079,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,1,1,3,01:24:22.798000,Finished,25,71,2024 Austrian Grand Prix
1080,81,O PIASTRI,PIA,piastri,McLaren,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,2,2,7,00:00:01.906000,Finished,18,71,2024 Austrian Grand Prix
1081,55,C SAINZ,SAI,sainz,Ferrari,E80020,ferrari,Carlos,Sainz,Carlos Sainz,3,3,4,00:00:04.533000,Finished,15,71,2024 Austrian Grand Prix
1082,44,L HAMILTON,HAM,hamilton,Mercedes,27F4D2,mercedes,Lewis,Hamilton,Lewis Hamilton,4,4,5,00:00:23.142000,Finished,12,71,2024 Austrian Grand Prix
1083,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,5,5,1,00:00:37.253000,Finished,10,71,2024 Austrian Grand Prix
1084,27,N HULKENBERG,HUL,hulkenberg,Haas F1 Team,B6BABD,haas,Nico,Hulkenberg,Nico Hulkenberg,6,6,9,00:00:54.088000,Finished,8,71,2024 Austrian Grand Prix
1085,11,S PEREZ,PER,perez,Red Bull Racing,3671C6,red_bull,Sergio,Perez,Sergio Perez,7,7,8,00:00:54.672000,Finished,6,71,2024 Austrian Grand Prix
1086,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,B6BABD,haas,Kevin,Magnussen,Kevin Magnussen,8,8,12,00:01:00.355000,Finished,4,71,2024 Austrian Grand Prix
1087,3,D RICCIARDO,RIC,ricciardo,RB,6692FF,rb,Daniel,Ricciardo,Daniel Ricciardo,9,9,11,00:01:01.169000,Finished,2,71,2024 Austrian Grand Prix
1088,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,10,10,13,00:01:01.766000,Finished,1,71,2024 Austrian Grand Prix


In [13]:
resultado = df_procesado.loc[(df_procesado['BroadcastName'] == 'G RUSSELL') & (df_procesado['Race'] == '2024 Austrian Grand Prix')]
resultado

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,Position,ClassifiedPosition,GridPosition,Time,Status,Points,Laps,Race
1079,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,1,1,3,01:24:22.798000,Finished,25,71,2024 Austrian Grand Prix
1159,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,1,1,3,01:24:22.798000,Finished,25,71,2024 Austrian Grand Prix


In [14]:
# Identifica y maneja duplicados
print(f"Registros duplicados encontrados: {df_procesado.duplicated().sum()}")
# Si hay duplicados, decide cómo manejarlos
df_procesado = df_procesado.drop_duplicates()
print(f"Total de registros después de limpieza: {len(df_procesado)}")


Registros duplicados encontrados: 80
Total de registros después de limpieza: 1836


Se eliminaron los valores duplicados ya que estos ocurrieron durante la obtención de datos, ya que se tenia que pausar cada cierta cantidad de llamadas al API debido al limite de llamadas por hora que tiene la libreria de FastF1.

## 5. Diseño del Esquema de Base de Datos
Primero se realizara el esquema de base de datos para poder hacer el Feature Engineering mas sencillo al tener PKs y FKs

### 5.1 Diseño Conceptual

Número de Tablas: 2 tablas

Tabla Principal: Resultados
- Propósito: contendra cuanto tiempo se demoro un conductor en acabar cada carrera y la diferencia de tiempo con los demas conductores
- Columnas clave: 
    - TotalRaceTime: Tiempo que se demoro cada conductor en completar la carrera ("DNF" si no completo la carrera)
    - Position: Posición en la que quedo el conductor
    - PositionDifference: Diferencia entre la posición inicial y final del conductor

    
Tabla Secundaria 1: Conductores
- Propósito: Contiene informacion sobre cada conductor
- Relación con tabla principal: Proporciona su ID como FK para la tabla Resultados. Obtenida al normalizar el dataframe original.

### 5.2 Justificación del Diseño

Se usaron 2 tablas debido a la normalización del dataframe original. Esto se debe a la primera forma normal (1FN) que elimina grupos repetitivos y asegura que cada celda tenga un valor único.
Debido al uso de FKs el agente tendra mayor facilidad al razonar ya que no tendra que hacer calculos para encontrar valores especificos sobre alguna carrera, conductor o equipo y solo tendra que usar sus IDs.

### 5.3 Nomenclatura de Columnas

Tabla Conductores:
- conductor_id (INTEGER): Identificador único del conductor
- DriverNumber (INTEGER): Numero que uso el conductor en la carrera
- FullName (TEXT): Nombre y primer apellido del conductor
- TeamName (TEXT): Equipo al que pertenece el conductor


Tabla Resultados:
- conductor_id (INTEGER): FK del conductor
- Race(TEXTO): Nombre de la carrera
- Position (INTEGER): Posición en la que quedo el conductor
- GridPosition (INTEGER): Posición inicial del conductor
- PositionDifference (INTEGER): Diferencia entre la posición inicial y final del conductor
- TotalRaceTime (TEXTO): tiempo que se demoro el conductor en completar la carrera ("DNF" si no la completo)
- Time (TEXTO): El tiempo del ganador de la carrera y la diferencia de cada conductor con el ganador ("DNF"si no completo la carrera).
- Status (TEXTO): Estado en la que el conductor termino la carrera o causa del DNF.
- Points (INTEGER): Puntos obtenidos por el conductor en la carrera
- Laps (INTEGER): Vueltas completadas por el conductor

### 5.4 Decisiones de Normalización vs Desnormalización

El dataframe tenia varios valores que se repetian y dificultaban el trabajo, debido a eso se realizo la normalizacion. Esto permitio generar FKs que ayudan a hacer calculos y eliminar información que se repetia en la tabla.

## 4. Feature Engineering

### Diseño de tablas

Dividir el Dataframe en las tablas que se usaran para la base de datos.

In [15]:
#Tabla conductores
# Selecciona columnas del conductor y elimina duplicados
df_conductores = df_procesado[['DriverNumber', 'FullName','TeamName']].drop_duplicates().reset_index(drop=True)

# Crea una columna ID para usarla como Primary Key (PK)
df_conductores.insert(0, 'conductor_id', range(1, len(df_conductores) + 1))

df_conductores.head()

,conductor_id,DriverNumber,FullName,TeamName
0,1,16,Charles Leclerc,Ferrari
1,2,55,Carlos Sainz,Ferrari
2,3,44,Lewis Hamilton,Mercedes
3,4,63,George Russell,Mercedes
4,5,20,Kevin Magnussen,Haas F1 Team


In [16]:
#Tabla resultados
# 1. Cruzamos el original con la tabla de conductores para obtener 'conductor_id'
df_resultados = pd.merge(df_procesado, df_conductores, on=['FullName', 'TeamName'])
print("aaaa")

# 2. Seleccionamos solo los IDs y los datos propios del resultado
df_resultados = df_resultados[['conductor_id', 'Position', 'GridPosition','Time','Status','Points','Laps','Race']]

df_resultados.head()

aaaa


,conductor_id,Position,GridPosition,Time,Status,Points,Laps,Race
0,1,1,1,01:37:33.584000,Finished,26,57,2022 Bahrain Grand Prix
1,2,2,3,00:00:05.598000,Finished,18,57,2022 Bahrain Grand Prix
2,3,3,5,00:00:09.675000,Finished,15,57,2022 Bahrain Grand Prix
3,4,4,9,00:00:11.211000,Finished,12,57,2022 Bahrain Grand Prix
4,5,5,7,00:00:14.754000,Finished,10,57,2022 Bahrain Grand Prix


### 4.1 Features Derivadas - Feature 1 - PositionDifference

In [17]:
#- PositionDifference (INTEGER): Diferencia entre la posición iniciel y final del conductor
#Crear PositionDifference restando la posición incial (GridPosition) con la posición final (Position).
df_resultados['PositionDifference'] = df_resultados['GridPosition'] - df_resultados['Position']
df_resultados.head()


,conductor_id,Position,GridPosition,Time,Status,Points,Laps,Race,PositionDifference
0,1,1,1,01:37:33.584000,Finished,26,57,2022 Bahrain Grand Prix,0
1,2,2,3,00:00:05.598000,Finished,18,57,2022 Bahrain Grand Prix,1
2,3,3,5,00:00:09.675000,Finished,15,57,2022 Bahrain Grand Prix,2
3,4,4,9,00:00:11.211000,Finished,12,57,2022 Bahrain Grand Prix,5
4,5,5,7,00:00:14.754000,Finished,10,57,2022 Bahrain Grand Prix,2


"PositionDifference" sirve para ver la diferencia entre la posición inicial (GridPosition) y la posición final (Position). 

Facilita preguntas relacionadas al desempeño durante la carrera. 
- Un conductor con un PositionDifference positivo significa que sobrepaso a otros conductores.
- Un coinductor con un PositionDifference negativo significa que fue sobrepasado a otros conductores.
- Un conductor con un PositionDifference con valor 0 significa que mantuvo su posición durante la carrera.

### 4.2 Features Derivadas - Feature 2 - TotalRaceTime

In [18]:
#- Time (TEXTO): tiempo que se demoro el conductor en completar la carrera ("DNF" si no la completo)
#df_resultados_copy = df_resultados.copy()
import pandas as pd

# 1. Convertir a Timedelta (los DNF pasan a ser NaT)
df_resultados['TimeTd'] = pd.to_timedelta(df_resultados['Time'], errors='coerce')

# 2. Mapear el tiempo del líder por carrera
leader_times = df_resultados[df_resultados['Position'] == 1].set_index('Race')['TimeTd']

# 3. Calcular tiempo total
def format_total_time(row):
    if pd.isna(row['TimeTd']):
        return "DNF"
    
    total = row['TimeTd'] if row['Position'] == 1 else leader_times[row['Race']] + row['TimeTd']
    
    # Formatear a string (HH:MM:SS.mmm)
    ts = total.total_seconds()
    hours, remainder = divmod(ts, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours):02}:{int(minutes):02}:{seconds:06.3f}"

df_resultados['TotalRaceTime'] = df_resultados.apply(format_total_time, axis=1)

df_resultados = df_resultados.drop(columns=['TimeTd'])
df_resultados.head()

,conductor_id,Position,GridPosition,Time,Status,Points,Laps,Race,PositionDifference,TotalRaceTime
0,1,1,1,01:37:33.584000,Finished,26,57,2022 Bahrain Grand Prix,0,01:37:33.584
1,2,2,3,00:00:05.598000,Finished,18,57,2022 Bahrain Grand Prix,1,01:37:39.182
2,3,3,5,00:00:09.675000,Finished,15,57,2022 Bahrain Grand Prix,2,01:37:43.259
3,4,4,9,00:00:11.211000,Finished,12,57,2022 Bahrain Grand Prix,5,01:37:44.795
4,5,5,7,00:00:14.754000,Finished,10,57,2022 Bahrain Grand Prix,2,01:37:48.338


La columna "Time" tenia un formato que muestra la diferencia de tiempo respecto al ganador, excepto para el primer lugar. La nueva medida "TotalRaceTime" muestra el tiempo de cada conductor sin importar en que posición quedo. Esta puede ser usada para varion analisís como calcular el tiempo promedio de vuelta o comparar resultados entre distintas carreras (e.g. El mismo Grand Prix pero en distintos años).

## 6. Implementación de la Base de Datos

### 6.1 Preparación Final del DataFrame


In [19]:
print("DataFrame final para carga a base de datos:")
print(df_conductores.head())
print(f"\nTotal de registros a cargar: {len(df_conductores)}")
print(df_resultados.head())
print(f"\nTotal de registros a cargar: {len(df_resultados)}")



DataFrame final para carga a base de datos:
   conductor_id  DriverNumber         FullName      TeamName
0             1            16  Charles Leclerc       Ferrari
1             2            55     Carlos Sainz       Ferrari
2             3            44   Lewis Hamilton      Mercedes
3             4            63   George Russell      Mercedes
4             5            20  Kevin Magnussen  Haas F1 Team

Total de registros a cargar: 54
   conductor_id  Position  GridPosition             Time    Status  Points  \
0             1         1             1  01:37:33.584000  Finished      26   
1             2         2             3  00:00:05.598000  Finished      18   
2             3         3             5  00:00:09.675000  Finished      15   
3             4         4             9  00:00:11.211000  Finished      12   
4             5         5             7  00:00:14.754000  Finished      10   

   Laps                     Race  PositionDifference TotalRaceTime  
0    57  2022 Bahra

### 6.2 Creación de la Base de Datos SQLite

In [20]:
# Define la ruta de tu base de datos
# Usa una ruta que sea fácil de encontrar y acceder
db_path = 'agente_sql_f1.db'
# Crea la conexión y carga los datos
engine = create_engine(f'sqlite:///{db_path}')
df_conductores.to_sql('conductores', engine, if_exists='replace', index=False)
df_resultados.to_sql('resultados', engine, if_exists='replace', index=False)
print(f"Base de datos creada exitosamente en: {db_path}")
print(f"Tabla creada: conductores")
print(f"Tabla creada: resultados")


Base de datos creada exitosamente en: agente_sql_f1.db
Tabla creada: conductores
Tabla creada: resultados


### 6.3 Verificación de la Carga

In [21]:
# Conecta a la base de datos y verifica que todo se cargó correctamente
conn = sqlite3.connect(db_path)
# Verifica el conteo de registros
query_conteo_conductores = "SELECT COUNT(*) as total FROM conductores"
query_conteo_resultados = "SELECT COUNT(*) as total FROM resultados"

print("Total de registros en la base de datos:")
print("Tabla conductores:")
print(pd.read_sql_query(query_conteo_conductores, conn))
print("Tabla resultados:")
print(pd.read_sql_query(query_conteo_resultados, conn))

# Verifica las primeras filas
query_muestra_conductores = "SELECT * FROM conductores LIMIT 5"
query_muestra_resultados = "SELECT * FROM resultados LIMIT 5"
print("\nPrimeras 5 filas en la base de datos:")
print("Tabla conductores:")
print(pd.read_sql_query(query_muestra_conductores, conn))
print("Tabla resultados:")
print(pd.read_sql_query(query_muestra_resultados, conn))

# Si tienes columnas numéricas, verifica agregaciones básicas
query_agregacion_points = "SELECT AVG(Points) as promedio FROM resultados"
query_agregacion_laps = "SELECT AVG(Laps) as promedio FROM resultados"
print("\nAgregación de prueba:")
print("Points:")
print(pd.read_sql_query(query_agregacion_points, conn))
print("Laps:")
print(pd.read_sql_query(query_agregacion_laps, conn))
conn.close()


Total de registros en la base de datos:
Tabla conductores:
   total
0     54
Tabla resultados:
   total
0   1869

Primeras 5 filas en la base de datos:
Tabla conductores:
   conductor_id  DriverNumber         FullName      TeamName
0             1            16  Charles Leclerc       Ferrari
1             2            55     Carlos Sainz       Ferrari
2             3            44   Lewis Hamilton      Mercedes
3             4            63   George Russell      Mercedes
4             5            20  Kevin Magnussen  Haas F1 Team
Tabla resultados:
   conductor_id  Position  GridPosition             Time    Status  Points  \
0             1         1             1  01:37:33.584000  Finished      26   
1             2         2             3  00:00:05.598000  Finished      18   
2             3         3             5  00:00:09.675000  Finished      15   
3             4         4             9  00:00:11.211000  Finished      12   
4             5         5             7  00:00:14.75400

Los datos se cargaron correctamente, la tabla conductores tiene 54 registros y la tabla resultados 1869. Estos numeros son los mismos que los datasets que se cargaron a la base de datos.
Las agregaciones tienen sentido. En el caso de Points, es correcto que el AVG no sea tan alto ya que solo los primeros 10 conductores en terminar la carrera obtienen puntos, mientras que el resto no. En el caso de Laps, es correcto que el AVG sea casi 55, ya que no todos los conductores completan el 100% de las vueltas debido a accidentes o problemas en sus carros.

## 7. Configuración del Agente SQL con LangChain


### 7.1 Configuración de API Key


In [23]:
# Configuración segura de credenciales de OpenAI
import os
# Intenta cargar desde Colab Secrets si estás en Colab
try:
 from google.colab import userdata
 openai_api_key = userdata.get('OPENAI_API_KEY')
 os.environ["OPENAI_API_KEY"] = openai_api_key
 print("✓ API Key cargada desde Colab Secrets")
except:
 # Si no estás en Colab, solicita la API key de forma interactiva
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Ingresa tu OpenAI API Key: ")
    print("✓ API Key configurada")

✓ API Key configurada


### 7.2 Inicialización del Agente SQL

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_openai import ChatOpenAI
# Conecta a tu base de datos SQLite
db = SQLDatabase.from_uri(f'sqlite:///{db_path}')
# Inicializa el modelo de lenguaje
llm = ChatOpenAI(
 model="gpt-3.5-turbo", # O "gpt-3.5-turbo" si prefieres
 temperature=0, # Temperatura 0 para respuestas más determinísticas
)
# Crea el agente SQL
agent_executor = create_sql_agent(
 llm=llm,
 db=db,
 agent_type="openai-tools",
 verbose=True, # Muestra el razonamiento del agente
)
print("✓ Agente SQL inicializado correctamente")


✓ Agente SQL inicializado correctamente


Seleccione GPT 3.5 Turbo ya que es un modelo mas rapido, los datos que hay en la base de datos son simples, entonces no es necesario tener un modelo con mejor razonamiento como GPT 4. El modelo tiene temperatura = 0 ya que se quiere medidas determinísticas, ya que se estan análizando datos reales con los que se mide el performance de los conductores. Respuestas no determinísticas podrian exagerar algunos datos y causar que se hagan cambios de conductores o a los carros y en ambos casos esto costaria miles o millones de dolares a las empresas.

### 7.3 Prueba de Conexión Básica

In [ ]:
# Prueba simple para verificar que el agente funciona
pregunta_prueba = "¿Cuántas filas hay en la tabla de conductores?"
print(f"Pregunta de prueba: {pregunta_prueba}")
print("\n" + "="*80 + "\n")
respuesta = agent_executor.invoke({"input": pregunta_prueba})
print("\nRespuesta del agente:")
print(respuesta['output'])


Pregunta de prueba: ¿Cuántas filas hay en la tabla de conductores?




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/
Invoking: `sql_db_query` with `{'query': 'SELECT COUNT(*) AS total_rows FROM conductores'}`


[(54,)]Hay 54 filas en la tabla de conductores.

> Finished chain.

Respuesta del agente:
Hay 54 filas en la tabla de conductores.


El agente respondio la pregunta correctamente

## 8. Pruebas del Agente SQL


### 8.1 Categorías de Prueba


- Lookup Queries (Buscar información específica)
- Filtering queries (filtrar registros)
- Ranking queries (ordenar resultados)
- Aggregation queries (sumas, conteos, métricas)
- Comparative queries (comparaciones o cambios)

### 8.2  Lookup Queries

#### Prueba 1

In [ ]:
pregunta_1 = "En que equipo corre Charles Leclerc"
respuesta_1 = agent_executor.invoke({"input": pregunta_1})
print(respuesta_1['output'])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/
Invoking: `sql_db_query` with `{'query': "SELECT TeamName FROM conductores WHERE FullName = 'Charles Leclerc'"}`


[('Ferrari',)]Charles Leclerc corre en el equipo Ferrari.

> Finished chain.
Charles Leclerc corre en el equipo Ferrari.


La respuesta es correcta, Charles Leclerc corre en el equipo Ferrari

#### Prueba 2

In [ ]:
pregunta_2 = "Que número de piloto tiene Lewis Hamilton"
respuesta_2 = agent_executor.invoke({"input": pregunta_2})
print(respuesta_2['output'])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/
Invoking: `sql_db_schema` with `{'table_names': 'resultados'}`



CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:

Correcto, Lewis Hamilton usa el número 44. Problema encontrado: El modelo respondio en ingles a un prompt en español

### 8.3 Filtering Queries

#### Prueba 3

In [ ]:
pregunta_3 = "Que pilotos no terminaron el Grand Prix de Ciudad de México 2023"
respuesta_3 = agent_executor.invoke({"input": pregunta_3})
print(respuesta_3['output'])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'resultados'}`



CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	18	57	2022 Bahrain Grand Prix	1	01:37:39.182	01:37:39.182
3	3	5	00:00:09.675000	Finished	15	57	2022 Bahrain Grand Prix	2	01:37:43.259	01:37:43.259
*/
Invoking: `sql_db_query` with `{'query': "SELECT DISTINCT conductor_id FROM resultados WHERE Status != 'Finished' AND Race = '2023 Mexico City Gr

In [ ]:
conn = sqlite3.connect(db_path)
query_muestra_conductores = "SELECT * FROM conductores WHERE conductor_id IN (25, 12, 23, 5, 18)"
print(pd.read_sql_query(query_muestra_conductores, conn))
conn.close()

   conductor_id  DriverNumber         FullName         TeamName
0             5            20  Kevin Magnussen     Haas F1 Team
1            12            18     Lance Stroll     Aston Martin
2            18            11     Sergio Perez  Red Bull Racing
3            23            14  Fernando Alonso     Aston Martin
4            25             2   Logan Sargeant         Williams


La respuesta menciona solo los IDs de los conductores en vez de los nombres. Una vez visto a quien se refiere los IDs que menciono, el modelo acerto que Fernando Alonso, Kevin Magnussen y Sergio Pérez no terminaron la carrera, pero tambien menciona que Logan Sargeant y Lance Stroll no terminaron la carrera. Esto se debe a que en sus Status sale "+ 1 Lap" y "+ 5 Lap" ya que ellos terminaron la carrera, pero el ganador les llevo 1 y 5 vueltas de ventaja.

#### Prueba 4

In [ ]:
pregunta_4 = "Que pilotos han sido parte del equipo Williams"
respuesta_4 = agent_executor.invoke({"input": pregunta_4})
print(respuesta_4['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/
Invoking: `sql_db_query` with `{'query': "SELECT DISTINCT FullName FROM conductores WHERE TeamName = 'Williams'"}`


[('Alexander Albon',), ('Nicholas Latifi',), ('Nyck De Vries',), ('Logan Sargeant',), ('Franco Colapinto',), ('Carlos Sainz',)]Los pilotos que han sido parte del equipo Williams son Alexander Albon, Nicholas Latifi, Nyck De Vries, Logan Sargeant, Franco Colapinto y Carlos Sainz.

> Finished chain.
Los pilotos que han sido parte del equipo Williams son Alexander Albon, Nicholas Latifi, Nyck De Vries, Logan Sargeant, Franc

Es correcto, los pilotos que han formado parte del equipo de Williams son:
- 2022: Alexander Albon, Nicholas Latifi y Nyck De Vries
- 2023: Alexander Albon y Logan Sargeant
- 2024: Alexander Albon, Logan Sargeant y Franco Colapinto
- 2025: Alexander Albon y Carlos Sainz

### 8.4 Ranking Queries

#### Prueba 5

In [ ]:
pregunta_5 = "Que piloto obtuvo mas puntos en 2025"
respuesta_5 = agent_executor.invoke({"input": pregunta_5})
print(respuesta_5['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores, resultados'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/


CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	1

El modelo menciono correctamente que el conductor que obtuvo mas puntos en 2025 fue Lando Norris, pero se equivoco en la cantidad de puntos, ya que menciono 394 puntos. En realidad Lnado Norris obtuvo 423 puntos.

#### Prueba 6

In [ ]:
pregunta_6 = "Quienes fueron los 5 primeros pilotos en el Grand Prix de Miami 2025"
respuesta_6 = agent_executor.invoke({"input": pregunta_6})
print(respuesta_6['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores, resultados'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/


CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	1

Correcto, los primeros pilotos en el Grand Prix de Miami 2025 fueron:
1. Oscar Piastri
2. Lando Norris
3. George Russell
4. Max Verstappen
5. Alexander Albon

### 8.5 Aggregation queries

#### Prueba 7

In [ ]:
pregunta_7 = "Cuantos puntos obtuvo Sergio Perez en 2022"
respuesta_7 = agent_executor.invoke({"input": pregunta_7})
print(respuesta_7['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'resultados'}`



CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	18	57	2022 Bahrain Grand Prix	1	01:37:39.182	01:37:39.182
3	3	5	00:00:09.675000	Finished	15	57	2022 Bahrain Grand Prix	2	01:37:43.259	01:37:43.259
*/
Invoking: `sql_db_query` with `{'query': "SELECT Points FROM resultados WHERE Race = '2022 Bahrain Grand Prix' AND conductor_id = 1"}`


[(26,)]S

Incorrecto, el modelo no mostro los puntos totales obtenidos por Sergio Perez en 2022 (305 puntos) y ademas halucino que gano 26 puntos en el GP de Bahrain cuando en realidad no gano puntos en esa carrera.

#### Prueba 8

In [ ]:
pregunta_8 = "Cuantos carreras se registraron en 2025"
respuesta_8 = agent_executor.invoke({"input": pregunta_8})
print(respuesta_8['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'resultados'}`



CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	18	57	2022 Bahrain Grand Prix	1	01:37:39.182	01:37:39.182
3	3	5	00:00:09.675000	Finished	15	57	2022 Bahrain Grand Prix	2	01:37:43.259	01:37:43.259
*/
Invoking: `sql_db_query` with `{'query': "SELECT DISTINCT Race FROM resultados WHERE Race LIKE '%2025%'"}`


[('2025 Australian Grand Prix',), ('

Incorrecto, en 2025 se registraron 24 carreras.

### 8.6 Comparative Queries

In [ ]:
pregunta_9 = "Cuantos puntos mas gano Lando Norris en 2025 a comparación de 2022"
respuesta_9 = agent_executor.invoke({"input": pregunta_9})
print(respuesta_9['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'resultados'}`



CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	18	57	2022 Bahrain Grand Prix	1	01:37:39.182	01:37:39.182
3	3	5	00:00:09.675000	Finished	15	57	2022 Bahrain Grand Prix	2	01:37:43.259	01:37:43.259
*/
Invoking: `sql_db_query` with `{'query': "SELECT Points, Race FROM resultados WHERE Race LIKE '%2022%' OR Race LIKE '%2025%'"}`


[(26, '2022 Bah

Incorrecto, en 2025 Lando Norris obtuvo 423 puntos mientras que en 2022 obtuvo 122. 

423 - 22 = 301

Lando Norris gano 301 puntos mas en 2025 en comparación con 2022.

#### Prueba 10

In [ ]:
pregunta_10 = "¿Quien obtuvo mas puntos durante todo 2025, Max Verstappen u Oscar Piastri?"
respuesta_10 = agent_executor.invoke({"input": pregunta_10})
print(respuesta_10['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores, resultados'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/


CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	1

Max Verstappen si obtuvo mas puntos que Oscar Piastri en 2025, pero la cantidad de puntos que menciona el agente son incorrectas.
- Max Verstappen: 421 puntos
- Oscar Piastri: 410 puntos

### 8.7 Pruebas de Variación Lingüística

#### Variación 1

In [ ]:
pregunta_var_1 = "Cuantos puntos obtuvo Checo Perez en 2022"
respuesta_var_1 = agent_executor.invoke({"input": pregunta_var_1})
print(respuesta_var_1['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'resultados'}`



CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	18	57	2022 Bahrain Grand Prix	1	01:37:39.182	01:37:39.182
3	3	5	00:00:09.675000	Finished	15	57	2022 Bahrain Grand Prix	2	01:37:43.259	01:37:43.259
*/
Invoking: `sql_db_query` with `{'query': "SELECT Points FROM resultados WHERE Race = '2022 Bahrain Grand Prix' AND conductor_id = 1"}`


[(26,)]C

El modelo genero el mismo query que en la pregunta original, pero sigue teniendo los mismos problemas

#### Variación 2


In [ ]:
pregunta_var_2 = "Que pilotos no terminaron el GP de CDMX en 2023"
respuesta_var_2 = agent_executor.invoke({"input": pregunta_var_2})
print(respuesta_var_2['output'])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


conductores, resultados
Invoking: `sql_db_schema` with `{'table_names': 'conductores, resultados'}`



CREATE TABLE conductores (
	conductor_id BIGINT, 
	"DriverNumber" BIGINT, 
	"FullName" TEXT, 
	"TeamName" TEXT
)

/*
3 rows from conductores table:
conductor_id	DriverNumber	FullName	TeamName
1	16	Charles Leclerc	Ferrari
2	55	Carlos Sainz	Ferrari
3	44	Lewis Hamilton	Mercedes
*/


CREATE TABLE resultados (
	conductor_id BIGINT, 
	"Position" BIGINT, 
	"GridPosition" BIGINT, 
	"Time" TEXT, 
	"Status" TEXT, 
	"Points" BIGINT, 
	"Laps" BIGINT, 
	"Race" TEXT, 
	"PositionDifference" BIGINT, 
	"Total_Race_Time" TEXT, 
	"TotalRaceTime" TEXT
)

/*
3 rows from resultados table:
conductor_id	Position	GridPosition	Time	Status	Points	Laps	Race	PositionDifference	Total_Race_Time	TotalRaceTime
1	1	1	01:37:33.584000	Finished	26	57	2022 Bahrain Grand Prix	0	01:37:33.584	01:37:33.584
2	2	3	00:00:05.598000	Finished	1

El modelo realizo un query similar pero en vez de mostrar solo IDs mostro los nombres. Mantiene los mismos errores.

## 9. Análisis de Limitaciones del Agente


### 9.1 Categorías de Queries Problemáticas

Categoría 1: Opéraciones matemáticas con datos filtrados

En todas las preguntas que se necesito que el agente filtre el dataset y luego realize alguna operación matemática hay errores (e.g. ¿Quien obtuvo mas puntos durante todo 2025, Max Verstappen u Oscar Piastri?). En los casos que ocurrio esto el modelo tuvo problemas filtrando los datos. En el caso del ejemplo, el modelo sumo todos los puntos obtenidos por Max Verstappen y Oscar Piastri sin importar el año

Categoria 2: Conductores que finalizaron carreras

Cuando se le pregunto "Que pilotos no terminaron el Grand Prix de Ciudad de México 2023" solo filtro a los que no tenian "Status" == "Finished", pero los Status "+x Lap" significan que el corredor finalizo la carrera, solo que el ganador le saco x vueltas de ventaja.

### 9.2 Mejoras Potenciales al Esquema


- Originalmente junte las columnas "year" y "raceName" en una sola, pero al ver los resultados creo que seria mejor no juntarlas. Esto le facilitaria filtrar por año al modelo.
- Añadiria una columna "Location" para que el modelo pueda filtrar por ciudades o paises, ya que hay paises que cuentan con mas de 1 Grand Prix y eso puede confundir al agente (e.g. Italia tiene el GP de italia y el GP de Emilia-Romaña).


### 9.3 Mejoras Potenciales a los Datos

- Actualmente no se puede responder preguntas sobre vueltas especificas (e.g. en que vueltas entraron a pits), pero el API de FastF1 cuenta con esta información, por lo que se podria añadir al dataset y crear una tabla nueva con estos datos.
- Para el proyecto limite a los años 2022, 2023, 2024 y 2025, pero FastF1 cuenta con informacion desde 1950. Esto permitiria tener data para comparar el desempeño de pilotos retirados con actuales o ver como se reducieron los tiempos cada año.

### 9.4 Limitaciones Fundamentales de la Aproximación

Un agente SQL no podria responder preguntas sobre como se sobrepasan los conductores o como giran en las curvas. Para responder estas preguntas se podria análizar imagenes y usar modelos de ML o DL.

## 10. Conclusiones y Reflexiones


### 10.1 Resumen del Proyecto

En este proyecto se desarrolló un agente SQL conversacional capaz de responder preguntas en lenguaje natural sobre un dataset de Fórmula 1 correspondiente a los años 2022, 2023, 2024 y 2025. Los datos fueron obtenidos a través del API de FastF1 y posteriormente procesados y almacenados en una base de datos SQLite.

El dataset original fue preprocesado y normalizado para crear una estructura de tablas que permitiera consultas más eficientes. Luego se configuró un agente SQL utilizando LangChain para que pudiera interpretar preguntas del usuario, generar consultas SQL automáticamente y recuperar la información relevante desde la base de datos

El sistema funciona correctamente para consultas simples (busqueda de resultados y filtrados simples). Sin embargo, se observaron limitaciones en preguntas que requirean filtrados mas complejos y operaciones matemáticas, donde el agente no aplicó correctamente los filtros antes de realizar calculos.

### 10.2 Aprendizajes Técnicos Clave


Durante el desarrollo del proyecto aprendi a implementar y usar agentes de SQL con LangChain. Ademas mejore mis habilidades de manipulación y preprocesamiento de datos en Python y de diseño de bases de datos.

### 10.3 Insights sobre Agentes SQL

Antes de empezar el proyecto asumía que los agentes SQL podrian responder preguntas complejas com mayor precisión. Debido a esto asumi que un modelo mas simple como GPT 3.5 turbo seria lo suficientemente capaz de resolver todos los prompts correctamente y por eso seleccione ese modelo. Ahora entiendo que tienen mas limitaciones y son capaces de halucinar datos por hacer un mal query.

### 10.4 Aplicabilidad a Contextos Reales

Un sistema de agentes SQL se podria usar cuando una empresa maneja una gran cantidad de datos, al implementar este tipo de sistemas podria permitir que usuarios o trabajadores tengan acceso a los datos sin necesidad de saber SQL.

En el contexto de F1, un sistema similar podria permitir a periodistas y analistas hacer preguntas sobre estadisticas de conductores y carreras sin necesidad de escribir querys en SQL.

Para un deployment en producción sería necesario:
- Mejorar la validación de las consultas generadas
- Usar un modelo con mejor capacidad de razonamiento
- Optimizar el esquema de la base de datos
- Incluir y guardar logs para monitorear el desempeño del agente

El valor para los stakeholders sería principalmente facilitar el acceso a los datos y reducir la barrera técnica para realizar análisis exploratorios. Ademas, permitiria que los mismos stakeholders busquen datos especificos usando lenguaje natural.

### 10.5 Desafíos y Cómo los Superaste


Inicialmente me parecio dificil la creación del feature "TotalRaceTime". Para esto realize varias pruebas pero me salian errores en el codigo. Al final, decidi primero normalizar las bases de datos para no tener que trabajar con todos los datos relacionados a conductor y poder enfocarme en solo los tiempos de la carrera.

### 10.6 Próximos Pasos Potenciales

Mis principales ideas para extender el proyecto son las siguientes:
- Añadir datos de mas años, esto permitiria hacer mas comparaciones entre conductores actuales y pasados.
- Añadir los datos de cada vuelta, esto permitiria tener los tiempos de cada vuelta, que tipo de llanta se uso y en que momento los conductores entraron a los pits.

Estas extensiones permitirian comparar el rendimiento de los conductores y equipos entre mas temporadas ademas de poder analizar las estrategias de cada equipo a la hora de cambiar las llantas y cual tipo seleccionar.

## Referencias


- F1 - the official home of Formula 1® racing. Formula 1® - The Official F1® Website. (n.d.-a). https://www.formula1.com/en/results/2022/drivers 
- F1 - the official home of Formula 1® racing. Formula 1® - The Official F1® Website. (n.d.-b). https://www.formula1.com/en/results/2025/drivers 
- Schäfer, P. (n.d.). Introduction to FASTF1#. Introduction to FastF1 - FastF1 (3.8.1). https://docs.fastf1.dev/index.html 
- Wikimedia Foundation. (2025a, April 18). Anexo:gran premio de baréin de 2022. Wikipedia. https://es.wikipedia.org/wiki/Anexo:Gran_Premio_de_Bar%C3%A9in_de_2022 
- Wikimedia Foundation. (2025b, December 12). Anexo:gran premio de miami de 2025. Wikipedia. https://es.wikipedia.org/wiki/Anexo:Gran_Premio_de_Miami_de_2025 
- Wikimedia Foundation. (n.d.). Anexo:gran premio de la ciudad de méxico de 2023. Wikipedia. https://es.wikipedia.org/wiki/Anexo:Gran_Premio_de_la_Ciudad_de_M%C3%A9xico_de_2023#Resultados 
- Williams en LA F1: Historia, Trayectoria Y Pilotos del Equipo. (n.d.). https://www.caranddriver.com/es/formula-1/equipos/a3486/williams/ 